In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, f_oneway

# Load data
churn_data = pd.read_csv('churn_data_with_features.csv')

print("="*70)
print("STATISTICAL VALIDATION: Does Competitor Threat Predict Churn?")
print("="*70)

# SECTION 1: Chi-Square Test
contingency = pd.crosstab(
    churn_data['category_threat_tier'],
    churn_data['Churn_binary']
)

print("\n✅ Contingency Table (Threat Tier × Churn):")
print(contingency)

chi2, p_value, dof, expected = chi2_contingency(contingency)

print(f"\n✅ Chi-Square Test:")
print(f"  χ² = {chi2:.2f}")
print(f"  p-value = {p_value:.6f}")
print(f"  dof = {dof}")
print(f"  Result: {'SIGNIFICANT (p < 0.05)' if p_value < 0.05 else 'NOT SIGNIFICANT'}")

# SECTION 2: Churn Rates by Tier
print(f"\n✅ Churn Rates by Threat Tier:")
for tier in ['Low', 'Medium', 'High']:
    tier_data = churn_data[churn_data['category_threat_tier'] == tier]
    churn_rate = tier_data['Churn_binary'].mean()
    count = len(tier_data)
    print(f"  {tier:8s}: {churn_rate:.1%} ({count} customers)")

# SECTION 3: ANOVA
print(f"\n✅ ANOVA Test (Churn Variance Across Threat Tiers):")
tier_churn_data = []
for tier in ['Low', 'Medium', 'High']:
    tier_data = churn_data[churn_data['category_threat_tier'] == tier]['Churn_binary']
    if len(tier_data) > 0:
        tier_churn_data.append(tier_data)

if len(tier_churn_data) >= 2:
    f_stat, p_value_anova = f_oneway(*tier_churn_data)
    print(f"  F-statistic = {f_stat:.2f}")
    print(f"  p-value = {p_value_anova:.6f}")
    print(f"  Result: {'SIGNIFICANT' if p_value_anova < 0.05 else 'NOT SIGNIFICANT'}")

# SECTION 4: Segment × Threat Analysis
print(f"\n{'='*70}")
print("SEGMENT × THREAT TIER ANALYSIS")
print(f"{'='*70}")

segment_analysis = []

for segment in ['Budget', 'Mid-tier', 'Premium', 'Enterprise']:
    segment_data = churn_data[churn_data['segment'] == segment]
    
    if len(segment_data) == 0:
        continue
    
    print(f"\n{segment.upper()} ({len(segment_data)} customers):")
    segment_detail = {'Segment': segment}
    
    for threat in ['Low', 'Medium', 'High']:
        threat_data = segment_data[segment_data['category_threat_tier'] == threat]
        
        if len(threat_data) == 0:
            continue
        
        churn_rate = threat_data['Churn_binary'].mean()
        count = len(threat_data)
        avg_spend = threat_data['MonthlyCharges'].mean()
        arr = count * avg_spend * 12
        
        segment_detail[f'{threat}_Churn'] = churn_rate
        segment_detail[f'{threat}_Customers'] = count
        segment_detail[f'{threat}_ARR'] = arr
        
        print(f"  {threat:8s}: {churn_rate:.1%} | {count:>4d} customers | ${arr:>12,.0f} ARR")
    
    segment_analysis.append(segment_detail)

results_df = pd.DataFrame(segment_analysis)
results_df.to_csv('segment_threat_analysis.csv', index=False)

print(f"\n✅ Saved segment analysis to segment_threat_analysis.csv")